In [1]:
import random

from rdkit import Chem

from stereomolgraph import AtomId, Bond, MolGraph
from stereomolgraph.experimental.canon import canon_atom_num
from stereomolgraph.ipython import View2D

In [2]:
def shuffle_graph(g: MolGraph) -> MolGraph:
    random_atom_ids: list[AtomId] = random.sample(range(len(g)), len(g))

    old_new_mapping: dict[AtomId, AtomId] = {}
    new_old_mapping: dict[AtomId, AtomId] = {}

    new_g = g.__class__()

    for atom, atom_type in random.sample(list(zip(g.atoms, g.atom_types)), len(g)):
        new_atom_id = random_atom_ids.pop()
        old_new_mapping[atom] = new_atom_id
        new_old_mapping[new_atom_id] = atom
        new_g.add_atom(old_new_mapping[atom], atom_type)

    for a1, a2 in random.sample(list(g.bonds), len(g.bonds)):
        new_g.add_bond(old_new_mapping[a1], old_new_mapping[a2])

    return new_g


In [3]:
def graph_to_set(g: MolGraph) -> frozenset[tuple[AtomId, int] | Bond]:
    ret: set[tuple[AtomId, int] | Bond] = {
        (a, t) for a, t in zip(g.atoms, g.atom_types)
    }
    for bond in g.bonds:
        ret.add(bond)
    return frozenset(ret)

In [4]:
def smiles2graph(smiles: str) -> MolGraph:
    rdmol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    # mg = MolGraph.from_rdmol(rdmol)
    smg = MolGraph.from_rdmol(rdmol)
    return smg

In [5]:
def test_canon_atom_num(g: MolGraph, n: int = 10) -> bool:
    view = View2D(show_atom_numbers=True)
    unique_graphs = set()
    canon_results: list[tuple[dict, MolGraph]] = []
    for i in range(n):
        shuffled_g = shuffle_graph(g)
        canon_id_mapping = canon_atom_num(shuffled_g)
        canon_graph = shuffled_g.relabel_atoms(canon_id_mapping)
        unique_graphs.add(graph_to_set(canon_graph))
        assert {len(ug) for ug in unique_graphs} == {len(g.atoms) + len(g.bonds)}
        canon_results.append((dict(canon_id_mapping), canon_graph))
    if len(unique_graphs) != 1:
        print(f"FAIL: {len(unique_graphs)} unique canonical forms found!")
        print("Original graph:")
        view(g)
        for i, (mapping, canon_graph) in enumerate(canon_results):
            print(f"Shuffle {i + 1}: atom mapping = {mapping}")
            view(canon_graph)
    return len(unique_graphs) == 1

In [6]:
difficult_graphs = (
    # "C1OC23COC45COC11COC67COC8(COC9(CO2)COC(CO1)(CO6)OCC(CO9)(OC4)OCC(CO5)(OC7)OC8)OC3",
    # # this is crazy slow because of a lot "equivalent atoms"
    "C12C3C4C5C1C6C7C2C8C3C6C5C8C74",  # peterson graph G(7,2)
    "C1(C2C3C4C15)C6C7C2C8C3C9C%10C4C%11C5C6C%12C%11C%10C%13C%12C7C8C9%13",
    "C1CC2CCC1CCC3CCC(CC3)CC2",
    "C12C3C1C4C5C4C5C23",
    "C1C2CC3CC1CC(C2)C3",  # adamantane
    "C12C3C4C1C5C2C3C45",  # cubane
)

Examples from: 
Krotko, D.G. Atomic ring invariant and Modified CANON extended connectivity algorithm for symmetry perception in molecular graphs and rigorous canonicalization of SMILES. J Cheminform 12, 48 (2020). https://doi.org/10.1186/s13321-020-00453-4

In [7]:
# Read SMILES from text file
def load_smiles_from_txt(filepath: str) -> list[str]:
    """Load SMILES strings from a text file"""
    with open(filepath, "r") as f:
        smiles_list = [line.strip() for line in f if line.strip()]
    print(f"Loaded {len(smiles_list)} SMILES from {filepath}")
    return smiles_list


# Load SMILES from file
smiles_file = load_smiles_from_txt("./smiles.txt")
smiles_list = [
    smiles_file[i - 1].split()[0]
    for i, _ in enumerate(smiles_file, start=1)
    # if i not in ids_to_skip
]

Loaded 300 SMILES from ./smiles.txt


In [8]:
for smiles in smiles_list:
    g = smiles2graph(smiles)

    try:
        assert test_canon_atom_num(g), smiles
        print(f"PASS: {smiles}")
    except AssertionError as e:
        print(e)
        print(smiles)
        break

PASS: C1[C@]2(CCCC1)CC=CC2
PASS: C1=CCC=CC=C1
PASS: C[S@@](=O)C
PASS: C1=CC=C2C(=C1)CCC=C2
PASS: OC=1C=CC=CC1[C@H](C2=CC=CC=C2O)O
PASS: C1[C@H]2C[C@H]3C[C@@H]1C[C@@H](C2)C3
PASS: O[C@@H](C=1C=CC=C[13CH]1)C=2[13CH]=CC=CC2
PASS: C=1[C@@H]2C=C[C@H](C1)C=C2
PASS: [C@@H]12[C@H]3[C@H]4[C@@H]1[C@@H]5[C@H]4[C@H]3[C@H]25
PASS: C=1C=2C(=C3C=4C5=C(C=CC4C=CC3=CC2)C=CC=C5)C6=C(C1)C=CC=C6
PASS: C=1C=2C(=C3C=4C5=C(C=CC4C=CC3=CC2)C=CC=C5)C6=C(C1)C=CC=C6
PASS: C1=C2CC=C3[C@]24C(CC=C4C1)=CC3
PASS: CC\C(\C(\C)=N\O)=N\O
PASS: [O-][N+](C=1C=CC(=CC1)[S@@](OCC)=O)=O
PASS: [SiH3][C@]([GeH3])(OC)SC
PASS: O=C([C@H]([C@H]([C@@H](C(O)=O)Cl)Cl)Cl)O
PASS: O=C([C@@H]([C@H]([C@H](C(O)=O)Cl)Cl)Cl)O
PASS: C1[C@@H]2C=C[C@H](CCCCCC\C=C\C1)CC2
PASS: C1CCCCC\C=C\C1
PASS: CCCCCCCCCC/C(=C(\C#N)/Br)/I
PASS: C[C@H](CC)O
PASS: C[C@](CC)([2H])O
PASS: O=C1CC=2C(C=3C(C1)=CC=CC3[N+]([O-])=O)=C(C=CC2)[N+]([O-])=O
PASS: C[C@H]1[C@@H]2CC[C@H]1[C@H](C2=O)Br
PASS: FC(CN1[C@@H](C2=CC=CS2)CCC1)(F)F
PASS: O[C@@H]/1CC/C=C\CC[C@H](\C=C1)C(C)